In [26]:
from datetime import datetime, timedelta
import requests

# NYC taxi data is published with ~2 month delay
# So we go back 3 months to be safe
today = datetime.now()
target_date = today.replace(day=1) - timedelta(days=90)

year = target_date.strftime("%Y")
month = target_date.strftime("%m")

url = f"https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_{year}-{month}.parquet"
filename = f"yellow_taxi_{year}_{month}.parquet"

print(f"Downloading: {url}")

# Download
response = requests.get(url)

# Check if valid
if response.status_code == 200 and len(response.content) > 10000:
    with open(f"/lakehouse/default/Files/{filename}", "wb") as f:
        f.write(response.content)
    print(f"✅ Downloaded: {filename} ({len(response.content)/1024/1024:.1f} MB)")
    
    # Read with Spark
    df_yellow = spark.read.parquet(f"Files/{filename}")
    print(f"Total trips: {df_yellow.count()}")
else:
    print(f"❌ Data not available yet for {year}-{month}")
    print("Using existing January 2023 data instead")
    df_yellow = spark.read.parquet("Files/yellow_taxi_2023.parquet")
    print(f"Total trips: {df_yellow.count()}")

StatementMeta(, a944d76e-7b2a-49d5-b5c3-0e33ef9ec8e9, 34, Finished, Available, Finished, False)

Downloading: https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2026-01.parquet
✅ Downloaded: yellow_taxi_2026_01.parquet (61.2 MB)
Total trips: 3724889


In [27]:
# Load NYC Yellow Taxi data from Lakehouse
df = spark.read.parquet("Files/sample_datasets/nyc_taxi_yellow_2020_1.parquet")

# Show first 10 rows
df.show(10)

# Show how many rows we have
print(f"Total rows: {df.count()}")

# Show column names
print(f"Columns: {df.columns}")

StatementMeta(, a944d76e-7b2a-49d5-b5c3-0e33ef9ec8e9, 35, Finished, Available, Finished, False)

+--------+-------------------+-------------------+--------------+------------+------------+------------+--------+--------+------+------+----------+---------------+-----------+----------+-----+------+--------------------+---------+-----------+-----------+
|vendorID| tpepPickupDateTime|tpepDropoffDateTime|passengerCount|tripDistance|puLocationId|doLocationId|startLon|startLat|endLon|endLat|rateCodeId|storeAndFwdFlag|paymentType|fareAmount|extra|mtaTax|improvementSurcharge|tipAmount|tollsAmount|totalAmount|
+--------+-------------------+-------------------+--------------+------------+------------+------------+--------+--------+------+------+----------+---------------+-----------+----------+-----+------+--------------------+---------+-----------+-----------+
|       2|2020-01-03 03:57:12|2020-01-03 04:02:37|             1|        1.03|          97|          97|    NULL|    NULL|  NULL|  NULL|         1|              N|          1|       6.0|  1.0|   0.5|                 0.3|     1.95|     

In [28]:
# Load NYC Green Taxi data too
df_green = spark.read.parquet("Files/sample_datasets/nyc_taxi_green_2020_11.parquet")
print(f"Green taxi rows: {df_green.count()}")
print(f"Yellow taxi rows: {df.count()}")

StatementMeta(, a944d76e-7b2a-49d5-b5c3-0e33ef9ec8e9, 36, Finished, Available, Finished, False)

Green taxi rows: 2
Yellow taxi rows: 1


In [29]:
# Drop old tables to avoid schema conflicts
spark.sql("DROP TABLE IF EXISTS silver_taxi_trips")
spark.sql("DROP TABLE IF EXISTS gold_hourly_stats")
print("Old tables dropped!")

StatementMeta(, a944d76e-7b2a-49d5-b5c3-0e33ef9ec8e9, 37, Finished, Available, Finished, False)

Old tables dropped!


In [30]:
# SILVER LAYER - Clean the data
from pyspark.sql.functions import col, hour, dayofweek, month

# Rename columns to match consistent naming
df_yellow = df_yellow.toDF(*[c.lower() for c in df_yellow.columns])

# Remove bad data
df_silver = df_yellow.filter(
    (col("passenger_count") > 0) &
    (col("trip_distance") > 0) &
    (col("fare_amount") > 0) &
    (col("total_amount") > 0)
)

# Add useful columns
df_silver = df_silver.withColumn("pickup_hour", hour(col("tpep_pickup_datetime"))) \
                     .withColumn("pickup_day", dayofweek(col("tpep_pickup_datetime"))) \
                     .withColumn("pickup_month", month(col("tpep_pickup_datetime")))

print(f"Raw rows: {df_yellow.count()}")
print(f"Clean rows: {df_silver.count()}")
print(f"Removed: {df_yellow.count() - df_silver.count()} bad records")

StatementMeta(, a944d76e-7b2a-49d5-b5c3-0e33ef9ec8e9, 38, Finished, Available, Finished, False)

Raw rows: 3724889
Clean rows: 2551851
Removed: 1173038 bad records


In [31]:
# Save Silver layer to Lakehouse
df_silver.write.mode("overwrite").saveAsTable("silver_taxi_trips")
print("Silver layer saved!")

# GOLD LAYER - Business insights
from pyspark.sql.functions import avg, sum, count, round

df_gold = df_silver.groupBy("pickup_hour").agg(
    count("*").alias("total_trips"),
    round(avg("fare_amount"), 2).alias("avg_fare"),
    round(avg("trip_distance"), 2).alias("avg_distance"),
    round(sum("total_amount"), 2).alias("total_revenue")
).orderBy("pickup_hour")

df_gold.show(24)

# Save Gold layer
df_gold.write.mode("overwrite").saveAsTable("gold_hourly_stats")
print("Gold layer saved!")

StatementMeta(, a944d76e-7b2a-49d5-b5c3-0e33ef9ec8e9, 39, Finished, Available, Finished, False)

Silver layer saved!
+-----------+-----------+--------+------------+-------------+
|pickup_hour|total_trips|avg_fare|avg_distance|total_revenue|
+-----------+-----------+--------+------------+-------------+
|          0|      59298|   20.61|        4.07|   1821796.73|
|          1|      38350|   17.92|        3.39|   1043845.89|
|          2|      26128|   16.73|        3.06|    671265.81|
|          3|      17772|   18.02|        3.42|    480778.35|
|          4|      12511|   23.14|        4.88|    405932.67|
|          5|      16357|   29.09|        6.74|    623786.47|
|          6|      33385|   25.14|        5.57|   1102805.62|
|          7|      66472|    20.7|        4.09|    1899719.9|
|          8|      93196|   19.09|        3.42|   2519131.37|
|          9|     110984|   18.85|        4.15|    2992893.2|
|         10|     122142|   18.86|        3.49|   3311242.49|
|         11|     131515|   18.58|        3.05|   3523734.04|
|         12|     144954|   18.99|        3.14|   

In [32]:
# Fix schema conflict - drop and recreate tables cleanly
spark.sql("DROP TABLE IF EXISTS silver_taxi_trips")
spark.sql("DROP TABLE IF EXISTS gold_hourly_stats")
print("Tables dropped!")

# Recreate Silver table
df_silver.write.mode("overwrite").saveAsTable("silver_taxi_trips")
print("Silver recreated!")

# Recreate Gold table
df_gold.write.mode("overwrite").saveAsTable("gold_hourly_stats")
print("Gold recreated!")

StatementMeta(, a944d76e-7b2a-49d5-b5c3-0e33ef9ec8e9, 41, Finished, Available, Finished, False)

Tables dropped!
Silver recreated!
Gold recreated!
